In [ ]:
# Step 1: Install necessary libraries
!pip install transformers datasets

In [ ]:
# Step 2: Import libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
# Step 3: Load GPT-Neo model
model_name = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("✅ GPT-Neo-125M model loaded successfully!")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ GPT-Neo-125M model loaded successfully!


In [ ]:
train_df = pd.read_csv("/content/mmlu_subset_12k_renamed.csv")
test_df = pd.read_csv("/content/human_ranked.csv")

print(f"Training samples: {len(train_df)}, Test samples: {len(test_df)}")

Training samples: 2000, Test samples: 480


In [ ]:
def prepare_train_examples(example):
    prompt = f"Question: {example['question']}\nA. {example['choice_0']}\nB. {example['choice_1']}\nC. {example['choice_2']}\nD. {example['choice_3']}\nChoose the correct answer:"
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
    label = index_to_letter[example['answer']]

    inputs = tokenizer(prompt, truncation=True, padding="max_length", max_length=256)

    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']

    labels = [-100] * len(input_ids)
    label_token_id = tokenizer.encode(label, add_special_tokens=False)[0]
    labels[-1] = label_token_id

    return {
        'input_ids': list(input_ids),
        'attention_mask': list(attention_mask),
        'labels': list(labels)
    }

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(prepare_train_examples)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_gptneo",
    num_train_epochs=0.1,  #  Lower epochs
    max_steps=100,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./logs_gptneo",
    logging_steps=20,
    save_strategy="no",
    #evaluation_strategy="no",
    report_to="none",
    fp16=True
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

<ipython-input-8-8a3bc22b4849>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("\n=== Fine-tuning GPT-Neo-125M Started ===")
trainer.train()


=== Fine-tuning GPT-Neo-125M Started ===


Step,Training Loss
20,2.748900
40,2.289600
60,1.934600
80,1.848200
100,1.764500


TrainOutput(global_step=100, training_loss=2.1171786117553713, metrics={'train_runtime': 911.9816, 'train_samples_per_second': 0.219, 'train_steps_per_second': 0.11, 'total_flos': 26120709734400.0, 'train_loss': 2.1171786117553713, 'epoch': 0.1})

In [ ]:
# Save model
model.save_pretrained("./fine_tuned_gptneo")
tokenizer.save_pretrained("./fine_tuned_gptneo")
print("Fine-tuning GPT-Neo-125M complete!")

In [ ]:
# Reload fine-tuned model
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_gptneo")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_gptneo")
model.eval()
model = model.to(device)

In [ ]:
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]  # Take last token's logits
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())
    return choice_logits

In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]

    logits = get_logits(model, tokenizer, question, choices)

    predicted_choice_idx = int(np.argmax(logits))
    correct_choice_index = int(row['correct_answer']) if 'correct_answer' in row else -1

    results.append({
        'question': question,
        'model_name': "EleutherAI/gpt-neo-125M",
        'variant': "gptneo-small",  #  New variant name
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': choices[0],
        'option_1': choices[1],
        'option_2': choices[2],
        'option_3': choices[3]
    })




In [ ]:
# Save predictions
final_df = pd.DataFrame(results)
output_path = "/content/predictions_low_group_gptneo.csv"
final_df.to_csv(output_path, index=False)

print(f"Predictions saved at {output_path}")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load tokenizer and model
model_checkpoint = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token  # Avoid warnings

model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
model = model.to(device)

In [ ]:
# Define formatting function for training
def format_training_sample(example):
    prompt = f"Question: {example['question']}\n"
    for i in range(4):
        prompt += f"{chr(65+i)}. {example[f'option_{i}']}\n"
    prompt += "Choose the correct answer:"
    target = chr(65 + int(example['correct_answer']))
    return {"input_ids": tokenizer(prompt, truncation=True, padding="max_length", max_length=256)["input_ids"],
            "labels": tokenizer(target, truncation=True, padding="max_length", max_length=2)["input_ids"]}

In [ ]:
# Load training dataset (replace with actual path if needed)
train_df = pd.read_csv("/content/training_data.csv")
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(format_training_sample)


In [ ]:
# Define training arguments for variant 2
training_args = TrainingArguments(
    output_dir="./neo_model2_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=3e-5,
    logging_dir="./logs",
    save_total_limit=1,
    save_steps=200,
    logging_steps=50,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
# Start training
trainer.train()

In [ ]:
# Save fine-tuned model
model.save_pretrained("./fine_tuned_neo_model2")
tokenizer.save_pretrained("./fine_tuned_neo_model2")
print("Variant 2 fine-tuning complete!")

In [ ]:
# Reload model for evaluation
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_neo_model2")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_neo_model2").to(device)
model.eval()

In [ ]:
# Load test data
test_df = pd.read_csv("/content/human_ranked.csv")


In [ ]:
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())
    return choice_logits

In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]
    logits = get_logits(model, tokenizer, question, choices)
    predicted_choice_idx = int(np.argmax(logits))
    correct_choice_index = int(row['correct_answer']) if 'correct_answer' in row else -1

    results.append({
        'question': question,
        'model_name': "EleutherAI/gpt-neo-125M",
        'variant': "neo_model2",
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': choices[0],
        'option_1': choices[1],
        'option_2': choices[2],
        'option_3': choices[3]
    })


In [ ]:
final_df = pd.DataFrame(results)
output_path = "/content/predictions_variant2_gptneo.csv"
final_df.to_csv(output_path, index=False)
print(f"Predictions saved at {output_path}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_checkpoint = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
model = model.to(device)

In [ ]:
def format_training_sample(example):
    prompt = f"Question: {example['question']}\n"
    for i in range(4):
        prompt += f"{chr(65+i)}. {example[f'option_{i}']}\n"
    prompt += "Choose the correct answer:"
    target = chr(65 + int(example['correct_answer']))
    return {
        "input_ids": tokenizer(prompt, truncation=True, padding="max_length", max_length=256)["input_ids"],
        "labels": tokenizer(target, truncation=True, padding="max_length", max_length=2)["input_ids"]
    }

In [ ]:
train_df = pd.read_csv("/content/training_data.csv")
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(format_training_sample)

In [ ]:
training_args = TrainingArguments(
    output_dir="./neo_model3_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_dir="./logs",
    save_total_limit=1,
    save_steps=200,
    logging_steps=50,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("./fine_tuned_neo_model3")
tokenizer.save_pretrained("./fine_tuned_neo_model3")
print("Variant 3 fine-tuning complete!")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_neo_model3")
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_neo_model3").to(device)
model.eval()

In [ ]:
test_df = pd.read_csv("/content/human_ranked.csv")

In [ ]:
def get_logits(model, tokenizer, question, choices):
    prompt = f"Question: {question}\n" + "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]) + "\nChoose the correct answer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]
    choice_logits = []
    for letter in ['A', 'B', 'C', 'D']:
        token_id = tokenizer.encode(letter, add_special_tokens=False)[0]
        choice_logits.append(logits[0, token_id].item())
    return choice_logits

In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    question = row['question']
    choices = [row['option_0'], row['option_1'], row['option_2'], row['option_3']]
    logits = get_logits(model, tokenizer, question, choices)
    predicted_choice_idx = int(np.argmax(logits))
    correct_choice_index = int(row['correct_answer']) if 'correct_answer' in row else -1

    results.append({
        'question': question,
        'model_name': "EleutherAI/gpt-neo-125M",
        'variant': "neo_model3",
        'predicted_choice': predicted_choice_idx,
        'correct_choice_index': correct_choice_index,
        'logit_A': logits[0],
        'logit_B': logits[1],
        'logit_C': logits[2],
        'logit_D': logits[3],
        'option_0': choices[0],
        'option_1': choices[1],
        'option_2': choices[2],
        'option_3': choices[3]
    })


In [ ]:
final_df = pd.DataFrame(results)
output_path = "/content/predictions_variant3_gptneo.csv"
final_df.to_csv(output_path, index=False)
print(f"Predictions saved at {output_path}")